# **Maestría en Inteligencia Artificial Aplicada**

## **Proyecto Integrador - TC5035**

## Semana 7: Avance 5 - Modelo final

### Profesores
- Dra. Grettel Barceló Alonso
- Dr. Luis Eduardo Falcón Morales

### Asesor
- Dr. Gerardo Jesús Camacho González

### Patrocinadores
- Dr. Jorge Antonio Ascencio Gutiérrez
- Yanmei King Loeza

### Equipo 29
- Carolina Lucas Dophe  –  A01702450
- Juan Pablo López Sánchez   –   A01313663
- Víctor Hugo Soto Herrera   –   A01706446

### Fecha de entrega
Domingo 1 de marzo de 2026

### Tabla de contenido
[Introducción](#intro)

1. [Carga de librerías y configuración inicial](#config)
2. [Preparación del entorno](#entorno)
3. [Definición del esquema de validación temporal](#esquema)
4. [Definición de métricas de evaluación](#metricas)
5. [Entrenamiento y evaluación de **modelos base**](#modelos)
   1. [Exponential Smoothing](#5_1)
   2. [ARIMA](#5_2)
   3. [SARIMAX](#5_3)
   4. [SVR](#5_4)
   5. [GPR](#5_5)
   6. [MLP](#5_6)
   7. [LSTM](#5_7)
   8. [Prophet](#5_8)
6. [Entrenamiento y evaluación de **modelos de ensamble**](#ensamble)
   1. [Promedio simple (Simple Averaging Ensemble)](#6_1)
   2. [Promedio ponderado (Weighted Average Ensemble)](#6_2)
   3. [Apilamiento con meta-modelo lineal (Stacking Ensemble con Ridge Regression)](#6_3)
   4. [Apilamiento con meta-modelo no lineal (Stacking Ensemble con Random Forest)](#6_4)
   5. [Bagging con GPR (Ensamble Homogéneo)](#6_5)
7. [Comparación global de modelos](#comparacion)
8. [Interpretación del modelo final](#interpretacion)

[Conclusiones](#conclusiones)

### <a class="anchor" id="intro">Introducción</a>

Este notebook corresponde al **Avance 5 del proyecto**, cuyo objetivo es extender el análisis realizado en la etapa anterior mediante la construcción y evaluación de **modelos de ensamble** para la predicción de la producción de aguacate.

En el **Avance 4** se desarrolló un conjunto de modelos base de distinta naturaleza metodológica, incluyendo enfoques estadísticos y de aprendizaje automático: ARIMA, SARIMAX, SVR, GPR, MLP y LSTM, utilizando además un modelo **Naïve Estacional** como referencia para la métrica MASE. Los resultados mostraron que los modelos con mejor desempeño fueron **Gaussian Process Regression (GPR)** y **SARIMAX**, seguidos por **SVR** y **MLP**.

A partir de estos resultados, en esta etapa se explora la posibilidad de **mejorar la capacidad predictiva mediante técnicas de ensamble**, combinando la información de múltiples modelos para capturar distintos patrones presentes en la serie temporal.

#### **Objetivo de esta entrega**

Los objetivos principales de este notebook son:

* Construir **modelos de ensamble homogéneos y heterogéneos** utilizando los modelos base desarrollados previamente.
* Evaluar el desempeño de estos modelos utilizando las métricas definidas en la etapa anterior.
* Comparar los modelos de ensamble con los modelos individuales.
* Seleccionar el **modelo final** considerando tanto el desempeño predictivo como la coherencia con el problema de negocio.

#### **Nota sobre reutilización del código**

Las secciones iniciales de este notebook reutilizan la estructura y la mayoría del código desarrollado en el **Avance 4**, incluyendo la preparación de datos, el esquema de validación temporal, la definición de métricas y el entrenamiento de los modelos base.

Dado que estas etapas forman parte del **pipeline fundamental del problema de predicción**, se mantienen prácticamente sin modificaciones, aplicando únicamente **ajustes menores de refactorización** cuando es necesario para facilitar su integración con los modelos de ensamble desarrollados en esta etapa.

Para una descripción detallada de estas etapas metodológicas, se recomienda consultar el notebook correspondiente al **Avance 4**.

### <a class="anchor" id="config">1. Carga de librerías y configuración inicial</a>

En esta sección se cargan las librerías necesarias para la implementación de los distintos modelos, el cálculo de métricas y la visualización de resultados. Asimismo, se establecen configuraciones iniciales orientadas a garantizar reproducibilidad y claridad en el análisis.

In [ ]:
# Librerías base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import time

# Modelos clásicos de series de tiempo
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Modelos de Machine Learning
from sklearn.svm import SVR
from prophet import Prophet
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, DotProduct,
    ConstantKernel, WhiteKernel
)

# Modelos de ensamble
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge

# Utilidades para el preprocesamiento y evaluación
from sklearn.utils import resample
from sklearn.inspection import PartialDependenceDisplay

# Modelo LSTM (Deep Learning)
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping
from keras.initializers import RandomNormal
from keras.optimizers import Adam

# Métricas de evaluación
from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Gráficas de diagnóstico
from statsmodels.stats.diagnostic import acorr_ljungbox

In [ ]:
# Configuración visual
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Configuración general
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

warnings.filterwarnings('ignore')

In [ ]:
# Definición de rutas base
DATA_DIR = Path("data")
PROCESSED_DATA_DIR = DATA_DIR / "processed"

### <a class="anchor" id="entorno">2. Preparación del entorno</a>

En esta sección se cargan las librerías necesarias y se realiza la lectura del conjunto de datos correspondiente a la serie temporal de producción de aguacate. Asimismo, se verifica la estructura del dataset, el formato de las variables y la correcta interpretación de la variable temporal, con el fin de asegurar la consistencia del análisis posterior.

#### 2.1 Carga de datos procesados

In [ ]:
# Cargar datos procesados
X = pd.read_csv(PROCESSED_DATA_DIR / "X_final_mensual.csv")
y = pd.read_csv(PROCESSED_DATA_DIR / "y_mensual.csv")

# Convertir Fecha a datetime
X['Fecha'] = pd.to_datetime(X['Fecha'])
y['Fecha'] = pd.to_datetime(y['Fecha'])

# Asignar Fecha como índice
X = X.set_index('Fecha')
y = y.set_index('Fecha')

# Ordenar por fecha
X = X.sort_index()
y = y.sort_index()

# Extraer la variable objetivo
y = y.iloc[:, 0]  # Volumenproduccion

print("Datos cargados exitosamente")
print(f"\nForma de X: {X.shape}")
print(f"Forma de y: {y.shape}")

#### 2.2 Verificación estructural

In [ ]:
# Verificar alineación X-y
assert (X.index == y.index).all(), "Los índices de X e y no coinciden"

# Mostrar rango temporal
print(f"Rango temporal: {X.index.min()} a {X.index.max()}")
print(f"Número de observaciones: {len(X)}")
print(f"\nDimensiones:")
print(f"  X: {X.shape}")
print(f"  y: {y.shape}")

# Verificar valores faltantes
print(f"\nValores faltantes en X: {X.isnull().sum().sum()}")
print(f"Valores faltantes en y: {y.isnull().sum()}")

# Primeras y últimas filas
print(f"\nPrimeras observaciones:")
print(X.head(5))
print(f"\nÚltimas observaciones:")
print(X.tail(5))

### <a class="anchor" id="esquema">3. Definición del esquema de validación temporal</a>

Dado que el problema corresponde a una **serie temporal**, se emplea un esquema de partición que respeta el orden cronológico de los datos. Esto permite evitar fuga de información (*data leakage*) entre entrenamiento y prueba, asegurando que los modelos sean evaluados en condiciones consistentes con un escenario de predicción real.

#### 3.1 Split fijo (80/20)

Se implementa una partición cronológica donde el 80% inicial de las observaciones se destina al entrenamiento y el 20% final se reserva para evaluación.

In [ ]:
# Definir el punto de split (80% entrenamiento, 20% prueba)
split_idx = int(len(X) * 0.8)
train_date = X.index[split_idx]

# Crear subconjuntos
X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f"Split realizado en: {train_date}")
print(f"\nConjunto de entrenamiento:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  Período: {X_train.index.min()} a {X_train.index.max()}")

print(f"\nConjunto de prueba:")
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape}")
print(f"  Período: {X_test.index.min()} a {X_test.index.max()}")

#### 3.2 Exportación de conjuntos de datos

Con el fin de mantener consistencia y trazabilidad entre avances, los subconjuntos de entrenamiento y prueba se exportan nuevamente al directorio `data/processed`.

In [ ]:
# Exportar subconjuntos a data/processed
X_train.to_csv(PROCESSED_DATA_DIR / "X_train_temporal.csv")
X_test.to_csv(PROCESSED_DATA_DIR / "X_test_temporal.csv")
y_train.to_csv(PROCESSED_DATA_DIR / "y_train_temporal.csv", header=True, index=True)
y_test.to_csv(PROCESSED_DATA_DIR / "y_test_temporal.csv", header=True, index=True)

print("Subconjuntos exportados.")

### <a class="anchor" id="metricas">4. Definición de métricas de evaluación</a>

Para evaluar el desempeño de los modelos se utilizan diversas métricas de error, incluyendo **MAE, RMSE, MAPE y MASE**. La métrica MASE se calcula utilizando como referencia un modelo **Naïve Estacional**, lo que permite interpretar el desempeño de los modelos en relación con un baseline sencillo pero representativo para series temporales.

#### 4.1 Métricas utilizadas

Las siguientes métricas se definen **antes** de entrenar los modelos, para evitar sesgo en la selección:

1. **MAE (Mean Absolute Error)**
   - Fórmula: $\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$
   - Ventaja: Interpretable directamente en unidades originales
   - Uso: Evaluación práctica del error esperado

2. **RMSE (Root Mean Squared Error)**
   - Fórmula: $\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$
   - Ventaja: Penaliza errores grandes más severamente
   - Uso: Comparación entre modelos (sensible a outliers)

3. **MAPE (Mean Absolute Percentage Error)**
   - Fórmula: $\text{MAPE} = \frac{1}{n}\sum_{i=1}^{n}\left|\frac{y_i - \hat{y}_i}{y_i}\right| \times 100$
   - Ventaja: Independiente de escala, útil en contexto productivo
   - Uso: Evaluación de precisión relativa

4. **R² (Coeficiente de determinación)**
   - Fórmula: $R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$
   - Ventaja: Métrica complementaria para evaluar capacidad explicativa
   - Uso: Proporción de varianza explicada

5. **MASE (Mean Absolute Scaled Error)**
   - Fórmula: $MASE = \frac{MAE_{modelo(TEST)}}{MAE_{naïve(TRAIN)}}$
   - El denominador se calcula exclusivamente sobre el conjunto de entrenamiento utilizando un modelo naïve estacional, garantizando que la evaluación en test no incorpore información futura.
   - Escala el error absoluto del modelo respecto al error de un modelo ingenuo de referencia.
   - Ventaja: Permite comparación justa entre modelos y evita problemas de escala.
   - Uso: Benchmarking contra un baseline temporal (naïve estacional en este caso).
   - Interpretación:
     - MASE < 1: Mejor que el baseline
     - MASE = 1: Igual al baseline
     - MASE > 1: Peor que el baseline

In [ ]:
# Cálculo del denominador MASE
# Naïve estacional (lag=12 para mensual)
# Para series temporales, la métrica necesita seguir usando como denominador el error absoluto medio del modelo Naïve estacional.
# https://en.wikipedia.org/wiki/Mean_absolute_scaled_error
seasonality = 12

naive_train = y_train.shift(seasonality)
naive_train = naive_train.dropna()

mase_denom = mean_absolute_error(
    y_train[seasonality:],
    naive_train
)

print(f"Denominador MASE (train naive estacional): {mase_denom:.4f}")

In [ ]:
# Función para calcular métricas de evaluación
def calculate_metrics(y_true, y_pred, mase_denom, model_name="Model"):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)

    mape = np.nanmean(
        np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true))
    ) * 100

    r2 = r2_score(y_true, y_pred)

    mase = mae / mase_denom if mase_denom != 0 else np.inf

    return {
        'Modelo': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R²': r2,
        'MASE': mase
    }


# Función para diagnóstico de residuos
def residuals_diagnostic(y_true, y_pred, lags=20, titulo="Modelo"):

    # Convertir a array si vienen como Series
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    residuals = y_true - y_pred

    print(f"\n--- Diagnóstico de Residuos: {titulo} ---")
    print(f"Media residuos: {np.mean(residuals):.6f}")
    print(f"Varianza residuos: {np.var(residuals):.6f}")

    # Test Ljung-Box
    lb_test = acorr_ljungbox(residuals, lags=[lags], return_df=True)
    print("\nLjung-Box p-value:", lb_test["lb_pvalue"].values[0])

    # Gráficas de diagnóstico
    fig, axs = plt.subplots(2, 2, figsize=(12,8))

    # 1. Serie temporal residuos
    axs[0,0].plot(residuals)
    axs[0,0].axhline(0, linestyle="--")
    axs[0,0].set_title("Residuos en el tiempo")

    # 2. Histograma + densidad
    sns.histplot(residuals, kde=True, ax=axs[0,1])
    axs[0,1].set_title("Distribución de residuos")

    # 3. ACF
    plot_acf(residuals, lags=lags, ax=axs[1,0])

    # 4. Residuos vs predicción (para homocedasticidad)
    axs[1,1].scatter(y_pred, residuals)
    axs[1,1].axhline(0, linestyle="--")
    axs[1,1].set_title("Residuos vs Predicción")

    plt.tight_layout()
    plt.show()

    return residuals

In [ ]:
# Lista para almacenar resultados de todos los modelos
resultados_modelos = []
modelos_entrenados = {}

# Contenedor para los mejores dos modelos
resultados_modelos_top2 = []
modelos_entrenados_top2 = {}

print("Contenedor de resultados inicializado")

### <a class="anchor" id="modelos">5. Entrenamiento y evaluación de **modelos base**</a>

En esta sección se entrenan los modelos individuales considerados en la etapa anterior: **ARIMA, SARIMAX, SVR, GPR, MLP, LSTM y Prophet**. Cada modelo se ajusta utilizando el conjunto de entrenamiento y posteriormente se generan predicciones sobre el conjunto de prueba, las cuales se utilizan para calcular las métricas de desempeño correspondientes.

Los resultados obtenidos en esta etapa permiten identificar los modelos con mejor desempeño, que posteriormente serán utilizados como componentes en las estrategias de **ensamble** desarrolladas en las secciones siguientes.

> **`IMPORTANTE`**: Aunque todos los modelos base desarrollados en el Avance 4 se mantienen en el notebook para efectos de comparación, la construcción de los modelos de ensamble se realiza únicamente utilizando aquellos que demostraron mayor capacidad predictiva. En particular, los modelos **GPR**, **SARIMAX**, **MLP**, **SVR** y **Prophet** mostraron un desempeño significativamente superior al modelo de referencia.
>
> Por el contrario, **ARIMA** y **LSTM presentaron resultados cercanos al baseline**, lo que sugiere una capacidad limitada para capturar patrones adicionales en la serie. Incluir modelos con bajo desempeño en un ensamble puede introducir ruido en las predicciones agregadas, por lo que estos modelos se conservan únicamente como referencia en la comparación final.

#### <a class="anchor" id="5_1">5.1 Modelo de Referencia: Exponential Smoothing</a>

**Descripción:** El modelo Exponential Smoothing, implementado mediante suavizamiento exponencial triple, modela explícitamente tres componentes de la serie temporal: nivel, tendencia y estacionalidad. En este caso se utiliza una formulación aditiva con un período estacional de 12 meses, apropiado para series mensuales con patrón anual.

In [ ]:
# Modelo Exponential Smoothing
seasonal_periods = 12

es_model = ExponentialSmoothing(
    y_train,
    trend='add',          # tendencia aditiva
    seasonal='add',       # estacionalidad aditiva
    seasonal_periods=seasonal_periods
).fit()

# Predicción para el horizonte del test
tiempo_inicial = time.time()
y_es = es_model.forecast(len(y_test))
y_es = pd.Series(y_es.values, index=y_test.index)
tiempo_entrenamiento = time.time() - tiempo_inicial

print(f"Tiempo de entrenamiento Exponential Smoothing: {tiempo_entrenamiento:.2f} segundos\n")

In [ ]:
metrics_es = calculate_metrics(
    y_test,
    y_es,
    mase_denom,
    model_name="Exponential Smoothing"
)
metrics_es["Tiempo_Entrenamiento"] = tiempo_entrenamiento

print(metrics_es)

resultados_modelos.append(metrics_es)

modelos_entrenados['Exponential_Smoothing'] = {
    'modelo': es_model,
    'y_pred': y_es
}

In [ ]:
# Visualización: Modelo Exponential Smoothing
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    y_test.index,
    y_test.values,
    label='Real',
    linewidth=2,
    marker='o',
    markersize=3
)

ax.plot(
    y_test.index,
    y_es,
    label='Predicción Exponential Smoothing',
    linewidth=2,
    linestyle='--',
    marker='s',
    markersize=3
)

ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción')
ax.set_title('Modelo Exponential Smoothing: Real vs Predicción')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### <a class="anchor" id="5_2">5.2 ARIMA</a>

**Descripción:** El modelo **ARIMA (AutoRegressive Integrated Moving Average)** es un enfoque estadístico clásico para series de tiempo univariadas que modela la dependencia temporal a través de tres componentes: autoregresivo (AR), diferenciación (I) y promedio móvil (MA).

In [ ]:
# Realizar prueba ADF
adf_test = adfuller(y_train)
print("ADF test p-value: ", adf_test[1])

In [ ]:
# Aplicar diferenciación
y_train_diff = y_train.diff().dropna()
y_train_diff.plot()

In [ ]:
# Comprobar estacionalidad
adf_test = adfuller(y_train_diff)
print("ADF test p-value: ", adf_test[1])

In [ ]:
# Graficar ACF y PACF del dataset estacional
plot_acf(y_train_diff)
plot_pacf(y_train_diff)
plt.show()

In [ ]:
# Entrenamiento del modelo ARIMA
arima_model = ARIMA(y_train, order = (50,1,50))

tiempo_inicial = time.time()
arima_model = arima_model.fit()
tiempo_entrenamiento = time.time() - tiempo_inicial

# Despliegue de métricas predefinidas por la librería
print(arima_model.summary())
print(f"\nTiempo de entrenamiento ARIMA: {tiempo_entrenamiento:.2f} segundos")

In [ ]:
# Se obtienen los residuos
residuos = arima_model.resid[1:]

# Se grafican los residuos y la densidad
fig, ax = plt.subplots(1,2)
residuos.plot(title = "Residuos", ax = ax[0])
residuos.plot(title = "Densidad", ax = ax[1], kind = 'kde')
plt.show()

In [ ]:
# Comparación gráfica entre el modelo entrenado y los datos reales sobre el conjunto de entrenamiento
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_train, label='Real', linewidth=2, marker='o', markersize=3)
ax.plot(arima_model.fittedvalues, label='Predicción ARIMA', linewidth=2, linestyle='--', marker='s', markersize=3)
ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción')
ax.set_title('Predicciones sobre el conjunto de entrenamiento')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Realizar predicciones
forecast = arima_model.forecast(steps = len(y_test))

# Comparación gráfica entre el modelo entrenado y los datos reales sobre el conjunto de prueba
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_test, label='Real', linewidth=2, marker='o', markersize=3)
ax.plot(forecast, label='Predicción ARIMA', linewidth=2, linestyle='--', marker='s', markersize=3)
ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción')
ax.set_title('Predicciones sobre el conjunto de prueba')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Entrenamiento del modelo ARIMA de forma automática
auto_arima_model = pm.auto_arima(y_train, stepwise = False, seasonal = False)
auto_arima_model.order

In [ ]:
# Despliegue de métricas predefinidas por la librería
auto_arima_model.summary()

In [ ]:
# Se obtienen los residuos
auto_residuos = auto_arima_model.resid()

# Se grafican los residuos y la densidad
fig, ax = plt.subplots(1,2)
auto_residuos.plot(title = "Residuos", ax = ax[0])
auto_residuos.plot(title = "Densidad", ax = ax[1], kind = 'kde')
plt.show()

In [ ]:
# Realizar predicciones
auto_forecast = auto_arima_model.predict(n_periods = len(y_test))

# Comparación gráfica entre el modelo entrenado manualmente, automático y los datos reales sobre el conjunto de prueba
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_test, label='Real', linewidth=2, marker='o', markersize=3)
ax.plot(forecast, label='Predicción ARIMA', linewidth=2, linestyle='--', marker='s', markersize=3)
ax.plot(auto_forecast, label='Predicción Auto-ARIMA', linewidth=2, linestyle='--', marker='s', markersize=3)
ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción')
ax.set_title('Predicciones sobre el conjunto de prueba')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Alineación de índices, que por algún motivo podrían estar desalineados
common_index = y_test.index.intersection(forecast.index)

y_true = y_test.loc[common_index]
y_pred = forecast.loc[common_index]

arima_model_metrics = calculate_metrics(
    y_true=y_true,
    y_pred=y_pred,
    mase_denom=mase_denom,
    model_name="ARIMA"
)

# Calcular métricas de desempeño para el modelo ARIMA
arima_model_metrics = calculate_metrics(y_true = y_test, y_pred = forecast, mase_denom = mase_denom, model_name = "ARIMA")
arima_model_metrics["Tiempo_Entrenamiento"] = tiempo_entrenamiento

# Almacenar para comparación
resultados_modelos.append(arima_model_metrics)

modelos_entrenados['ARIMA'] = {
    'modelo': arima_model,
    'y_pred': forecast
}

arima_model_metrics

In [ ]:
# Calcular métricas de desempeño para el modelo auto-ARIMA
auto_arima_model_metrics = calculate_metrics(y_true = y_test, y_pred = auto_forecast, mase_denom = mase_denom, model_name = "Auto-ARIMA")
auto_arima_model_metrics["Tiempo_Entrenamiento"] = tiempo_entrenamiento

# Almacenar para comparación
resultados_modelos.append(auto_arima_model_metrics)

modelos_entrenados['Auto_ARIMA'] = {
    'modelo': auto_arima_model,
    'y_pred': auto_forecast
}

auto_arima_model_metrics

#### <a class="anchor" id="5_3">5.3 SARIMAX</a>

**Descripción:** El modelo **SARIMAX (Seasonal ARIMA with eXogenous variables)** extiende ARIMA al incorporar componentes estacionales explícitos y la posibilidad de incluir variables exógenas. En series mensuales con estacionalidad anual, el componente estacional permite modelar dependencias con rezagos de 12 períodos, mientras que las variables explicativas (`X_train`) pueden capturar efectos estructurales adicionales.

In [ ]:
# Entrenar modelo SARIMAX y obtener los mejores hiperparámetros
tiempo_inicial = time.time()
sarimax_model = pm.auto_arima(y = y_train, X = X_train, m = 12)
tiempo_entrenamiento = time.time() - tiempo_inicial

print(f"Tiempo de entrenamiento SARIMAX: {tiempo_entrenamiento:.2f} segundos")

In [ ]:
# Mejores hiperparámetros
sarimax_model.order

In [ ]:
# Despliegue de métricas predefinidas por la librería
sarimax_model.summary()

In [ ]:
# Se obtienen los residuos
sarimax_residuos = sarimax_model.resid()

# Se grafican los residuos y la densidad
fig, ax = plt.subplots(1,2)
sarimax_residuos.plot(title = "Residuos", ax = ax[0])
sarimax_residuos.plot(title = "Densidad", ax = ax[1], kind = 'kde')
plt.show()

In [ ]:
# Realizar predicciones
sarimax_forecast = pd.Series(sarimax_model.predict(n_periods = len(y_test), X = X_test))
sarimax_forecast.index = y_test.index

# Comparación gráfica entre el modelo entrenado y los datos reales sobre el conjunto de prueba
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_test, label='Real', linewidth=2, marker='o', markersize=3)
ax.plot(sarimax_forecast, label='Predicción SARIMAX', linewidth=2, linestyle='--', marker='s', markersize=3)
ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción')
ax.set_title('Predicciones sobre el conjunto de prueba')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calcular métricas de desempeño para el modelo SARIMAX
sarimax_model_metrics = calculate_metrics(y_true = y_test, y_pred = sarimax_forecast, mase_denom = mase_denom, model_name = "SARIMAX")
sarimax_model_metrics["Tiempo_Entrenamiento"] = tiempo_entrenamiento

# Almacenar para comparación
resultados_modelos.append(sarimax_model_metrics)

modelos_entrenados['SARIMAX'] = {
    'modelo': sarimax_model,
    'y_pred': sarimax_forecast
}

sarimax_model_metrics

In [ ]:
# Predicciones sobre el conjunto de entrenamiento
sarimax_model = modelos_entrenados["SARIMAX"]["modelo"]
y_pred_train_sarimax = sarimax_model.fittedvalues()
modelos_entrenados["SARIMAX"]["y_pred_train"] = y_pred_train_sarimax

#### <a class="anchor" id="5_4">5.4 SVR</a>

**Descripción:** El modelo **Support Vector Regression (SVR)** es una extensión de las Máquinas de Vectores de Soporte al problema de regresión. Su objetivo es encontrar una función que minimice el error dentro de un margen tolerado, manteniendo simultáneamente la máxima generalización.

In [ ]:
# Definir modelo inicial
svr_model_init = SVR(
                    kernel='rbf',
                    C=1,
                    epsilon=0.1,
                    gamma= 'scale'
                  )

# Entrenamiento
svr_model_init.fit(X_train, y_train)

# Predicción
y_svr_init = svr_model_init.predict(X_test)

In [ ]:
# Calcular métricas
metrics_svr_init = calculate_metrics(
                                    y_test,
                                    y_svr_init,
                                    mase_denom,
                                    model_name="SVR"
                                  )

metrics_svr_init

In [ ]:
# Definir modelo
svr_model = SVR(
                kernel='rbf',
                C=100,
                epsilon=0.01,
                gamma= 0.001
               )

# Entrenamiento
tiempo_inicial = time.time()
svr_model.fit(X_train, y_train)
tiempo_entrenamiento = time.time() - tiempo_inicial

print(f"Tiempo de entrenamiento SVR: {tiempo_entrenamiento:.2f} segundos")

# Predicción
y_svr = svr_model.predict(X_test)

In [ ]:
# Calcular métricas
metrics_svr = calculate_metrics(
                                y_test,
                                y_svr,
                                mase_denom,
                                model_name="SVR"
                               )
metrics_svr["Tiempo_Entrenamiento"] = tiempo_entrenamiento


# Almacenar para comparación
resultados_modelos.append(metrics_svr)

modelos_entrenados['SVR'] = {
                            'modelo': svr_model,
                            'y_pred': y_svr
                            }
metrics_svr

In [ ]:
# Predicciones sobre el conjunto de entrenamiento
y_pred_train_svr = modelos_entrenados["SVR"]["modelo"].predict(X_train)
modelos_entrenados["SVR"]["y_pred_train"] = y_pred_train_svr

In [ ]:
# Visualización: Modelo SVR Predicciones sobre el conjunto de prueba
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_test.index, y_test.values, label='Real', linewidth=2, marker='o', markersize=3)
ax.plot(y_test.index, y_svr, label='Predicción SVR', linewidth=2, linestyle='--', marker='s', markersize=3)
ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción')
ax.set_title('Modelo SVR: Real vs Predicción')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# SVR: Predicciones sobre conjunto de entrenamiento

y_train_pred_svr = svr_model.predict(X_train)

plt.figure(figsize=(14, 6))
plt.plot(y_train.index, y_train.values, label='Real', linewidth=2)
plt.plot(y_train.index, y_train_pred_svr, label='Predicción SVR', linestyle='--')
plt.xlabel('Fecha')
plt.ylabel('Volumen de Producción')
plt.title('SVR: Predicciones sobre el conjunto de entrenamiento')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### <a class="anchor" id="5_5">5.5 GPR</a>

**Descripción:** La **Gaussian Process Regression (GPR)** es un enfoque probabilístico no paramétrico que modela la serie como una distribución sobre funciones. A través de una función de covarianza (kernel), GPR captura la estructura de dependencia entre observaciones y genera predicciones junto con estimaciones explícitas de incertidumbre.

##### Selección de Kernel

In [ ]:
X_train_gpr = X_train.copy()
X_test_gpr = X_test.copy()
y_train_gpr = y_train.copy()
y_test_gpr = y_test.copy()

In [ ]:
# Agregar lag1
y_train_lag = y_train_gpr.shift(1)
X_train_gpr["lag1"] = y_train_lag

# Eliminar primera fila (NaN por lag)
X_train_gpr = X_train_gpr.iloc[1:]
y_train_gpr = y_train_gpr.iloc[1:]

# Test
X_test_gpr["lag1"] = y_test_gpr.shift(1)
X_test_gpr.iloc[0, X_test_gpr.columns.get_loc("lag1")] = y_train_gpr.iloc[-1]

In [ ]:
# Definir kernels base
kernels = {
    "Linear": DotProduct(),
    "RBF": RBF(),
    "Matern_1.5": Matern(nu=1.5),
    "Matern_2.5": Matern(nu=2.5),
    "RationalQuadratic": RationalQuadratic()
}

resultados_kernels = []

for nombre, kernel in kernels.items():

    print(f"\nEvaluando kernel: {nombre}")

    # Modelo GPR
    gpr = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=10,
        random_state=42
    )

    # Entrenamiento
    gpr.fit(X_train_gpr, y_train_gpr)

    # Predicción
    y_pred, y_std = gpr.predict(X_test_gpr, return_std=True)

    # Métricas
    mae = mean_absolute_error(y_test_gpr, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_gpr, y_pred))
    mape = np.mean(np.abs((y_test_gpr - y_pred) / y_test_gpr)) * 100

    # MASE respecto a naive ya calculado
    mase = mae / mase_denom if mase_denom != 0 else np.inf

    resultados_kernels.append({
        "Kernel": nombre,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "MASE": mase
    })

# Crear DataFrame comparativo
df_kernels = pd.DataFrame(resultados_kernels)

# Ranking combinado
df_kernels["Rank_MAE"] = df_kernels["MAE"].rank()
df_kernels["Rank_RMSE"] = df_kernels["RMSE"].rank()
df_kernels["Rank_MAPE"] = df_kernels["MAPE"].rank()
df_kernels["Rank_MASE"] = df_kernels["MASE"].rank()

df_kernels["Rank_Promedio"] = df_kernels[
    ["Rank_MAE", "Rank_RMSE", "Rank_MAPE", "Rank_MASE"]
].mean(axis=1)

df_kernels = df_kernels.sort_values("Rank_Promedio")

df_kernels

##### Entrenamiento y Predicción

In [ ]:
# Definir kernel final
kernel_final = (
    ConstantKernel(1.0, (1e-3, 1e3)) *
    (
        Matern(length_scale=1.0, nu=1.5) +
        RationalQuadratic(length_scale=1.0, alpha=1.0)
    ) +
    WhiteKernel(noise_level=1e-5)
)

# Crear modelo
gpr_model = GaussianProcessRegressor(
    kernel=kernel_final,
    n_restarts_optimizer=10,
    random_state=42
)

# Entrenar
tiempo_inicial = time.time()
gpr_model.fit(X_train_gpr, y_train_gpr)
tiempo_entrenamiento = time.time() - tiempo_inicial

print(f"Tiempo de entrenamiento GPR: {tiempo_entrenamiento:.2f} segundos\n")

# Predicción TEST (media + desviación estándar)
y_pred_gpr, y_std_gpr = gpr_model.predict(X_test_gpr, return_std=True)

print("Kernel optimizado:")
print(gpr_model.kernel_)

##### Visualizaciones

In [ ]:
# Intervalos de confianza
lower_bound = y_pred_gpr - 2 * y_std_gpr
upper_bound = y_pred_gpr + 2 * y_std_gpr

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(y_test_gpr.index, y_test_gpr.values,
        label='Real', linewidth=2, marker='o', markersize=3)

ax.plot(y_test_gpr.index, y_pred_gpr,
        label='Predicción GPR', linewidth=2, linestyle='--')

ax.fill_between(
    y_test_gpr.index,
    lower_bound,
    upper_bound,
    alpha=0.2,
    label='IC 95%'
)

ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción (normalizado)')
ax.set_title('GPR: Real vs Predicción con Incertidumbre')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Diagnóstico de residuos para GPR
residuos_gpr = residuals_diagnostic(y_test_gpr, y_pred_gpr, lags=24, titulo="GPR")

##### Métricas Finales

In [ ]:
# Calcular métricas
metrics_gpr = calculate_metrics(
    y_true=y_test_gpr,
    y_pred=y_pred_gpr,
    mase_denom=mase_denom,
    model_name="GPR"
)
metrics_gpr["Tiempo_Entrenamiento"] = tiempo_entrenamiento

# Guardar resultados
resultados_modelos.append(metrics_gpr)
print(metrics_gpr)

# Almacenar modelo
modelos_entrenados['GPR'] = {
    'modelo': gpr_model,
    'y_pred': y_pred_gpr,
    'y_std': y_std_gpr
}

In [ ]:
# Predicciones sobre el conjunto de entrenamiento
y_pred_train_gpr, y_std_train_gpr = modelos_entrenados["GPR"]["modelo"].predict(
    X_train_gpr,
    return_std=True
)
modelos_entrenados["GPR"]["y_pred_train"] = y_pred_train_gpr
modelos_entrenados["GPR"]["y_std_train"] = y_std_train_gpr
modelos_entrenados["GPR"]["train_index"] = X_train_gpr.index

In [ ]:
# Remover "lag1" para no afectar otros modelos
X_train_gpr = X_train_gpr.drop(columns=["lag1"])
X_test_gpr = X_test_gpr.drop(columns=["lag1"])

#### <a class="anchor" id="5_6">5.6 MLP</a>

**Descripción:** El **Multi-Layer Perceptron (MLP)** es una red neuronal artificial de tipo feedforward compuesta por capas ocultas y funciones de activación no lineales. A través del aprendizaje por retropropagación, el modelo ajusta pesos internos para aproximar relaciones complejas entre entradas y salida.

In [ ]:
# Definir modelo default
mlp_model_default = MLPRegressor(
                          hidden_layer_sizes=(100,),
                          activation='relu',
                          solver='adam',
                          alpha=0.0001,
                          learning_rate_init=0.001,
                          max_iter=500,
                          random_state=42
                          )

# Entrenamiento
mlp_model_default.fit(X_train, y_train)

# Predicción
y_mlp_default = mlp_model_default.predict(X_test)
# Métricas modelo inicial
metrics_mlp_default = calculate_metrics(
                                  y_test,
                                  y_mlp_default,
                                  mase_denom,
                                  model_name="MLP_default"
                              )


metrics_mlp_default

In [ ]:
# Modelo optimizado
mlp_model_optimizado = MLPRegressor(
                          hidden_layer_sizes=(150,),
                          activation='relu',
                          solver='adam',
                          alpha=0.0001,
                          learning_rate_init=0.001,
                          max_iter=800,
                          early_stopping=True,
                          random_state=42
                          )

# Entrenamiento
mlp_model_optimizado.fit(X_train, y_train)

# Predicción
y_mlp_optimizado = mlp_model_optimizado.predict(X_test)
# Métricas modelo inicial
metrics_mlp_optimizado = calculate_metrics(
                                  y_test,
                                  y_mlp_optimizado,
                                  mase_denom,
                                  model_name="MLP_optimizado"
                              )


metrics_mlp_optimizado


In [ ]:
def create_lags(X, y, n_lags):

    df = X.copy()  # Copia de las variables originales

    # Crear variables
    for lag in range(1, n_lags + 1):
        df[f"y_lag_{lag}"] = y.shift(lag)

    df["target"] = y  # Agregar el valor actual como objetivo

    df = df.dropna()  # Eliminar filas con NaN por los rezagos

    X_new = df.drop(columns=["target"])  # Variables predictoras
    y_new = df["target"]  # Variable objetivo

    return X_new, y_new

In [ ]:
# Evaluación del MLP con parámetros optimizados usando ventanas de 6 y 12
results = []

for n_lags in [6, 12]:

    # Crear ventanas en TRAIN
    X_train_lag, y_train_lag = create_lags(X_train, y_train, n_lags)


    # Unir últimos datos de TRAIN con TEST para tener historial
    X_test_full = pd.concat([X_train.tail(n_lags), X_test])
    y_test_full = pd.concat([y_train.tail(n_lags), y_test])

    # Crear ventanas en el bloque combinado
    X_test_lag, y_test_lag = create_lags(X_test_full, y_test_full, n_lags)

    # Quedarse solo con el periodo real de test
    X_test_lag = X_test_lag.loc[X_test.index]
    y_test_lag = y_test_lag.loc[y_test.index]

    mlp_model_lag_opt = MLPRegressor(
                                      hidden_layer_sizes=(150,),
                                      activation='relu',
                                      solver='adam',
                                      alpha=0.0001,
                                      learning_rate_init=0.001,
                                      max_iter=800,                 
                                      early_stopping=True,
                                      random_state=42
                                  )


    mlp_model_lag_opt.fit(X_train_lag, y_train_lag)
    y_pred_lag_opt = mlp_model_lag_opt.predict(X_test_lag)

    metrics = calculate_metrics(
                                  y_test_lag,
                                  y_pred_lag_opt,
                                  mase_denom,
                                  model_name=f"MLP_lag_{n_lags}_con_prametros_cambiados"
                              )

    results.append(metrics)

results

In [ ]:
# Definir modelo MLP final

# Modelo optimizado
mlp_model_optimizado = MLPRegressor(
                          hidden_layer_sizes=(150,),
                          activation='relu',
                          solver='adam',
                          alpha=0.0001,
                          learning_rate_init=0.001,
                          max_iter=800,
                          early_stopping=True,
                          random_state=42
                          )

# Entrenamiento
mlp_model_optimizado.fit(X_train, y_train)

# Predicción
y_mlp_optimizado = mlp_model_optimizado.predict(X_test)

In [ ]:
inicio = time.time()
mlp_model_optimizado.fit(X_train, y_train)
fin = time.time()
print("Tiempo MLP:", fin - inicio, "segundos")

In [ ]:
# Métricas modelo MLP final
metrics_mlp_optimizado = calculate_metrics(
                                  y_test,
                                  y_mlp_optimizado,
                                  mase_denom,
                                  model_name="MLP_optimizado"
                              )
metrics_mlp_optimizado["Tiempo_Entrenamiento"] = fin - inicio

# Guardar resultados
resultados_modelos.append(metrics_mlp_optimizado)

modelos_entrenados['MLP_optimizado'] = {
    'modelo': mlp_model_optimizado,
    'y_pred': y_mlp_optimizado
}

metrics_mlp_optimizado

In [ ]:
# Predicciones sobre el conjunto de entrenamiento
y_pred_train_mlp = modelos_entrenados["MLP_optimizado"]["modelo"].predict(X_train)
modelos_entrenados["MLP_optimizado"]["y_pred_train"] = y_pred_train_mlp

In [ ]:
# Visualización: Modelo MLP
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_test.index, y_test.values, label='Real', linewidth=2, marker='o', markersize=3)
ax.plot(y_test.index, y_mlp_optimizado, label='Predicción MLP', linestyle='--', marker='s', markersize=3)
ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción')
ax.set_title('Modelo MLP: Real vs Predicción')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
# Predicciones sobre el conjunto de entrenamiento
y_train_pred_mlp = mlp_model_optimizado.predict(X_train)
ax.plot(y_train.index, y_train.values, label='Real', linewidth=2)
ax.plot(y_train.index, y_train_pred_mlp, label='Predicción MLP TRAIN', linestyle='--', linewidth=2)
ax.set_xlabel('Fecha')
ax.set_ylabel('Volumen de Producción')
ax.set_title('MLP: Predicciones sobre el conjunto de entrenamiento')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### <a class="anchor" id="5_7">5.7 LSTM</a>

**Descripción:** Las **Long Short-Term Memory Networks (LSTM)** son un tipo de red neuronal recurrente diseñada específicamente para modelar dependencias temporales de largo plazo. A diferencia de redes feedforward, las LSTM incorporan celdas de memoria y mecanismos de compuertas que permiten retener o descartar información relevante a lo largo del tiempo.

##### Transformación a estructura secuencial

In [ ]:
# Se aplica la transformación log1p para estabilizar la varianza y reducir el efecto
# de la tendencia creciente en la serie, facilitando el aprendizaje de la LSTM.
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

train_df = X_train.copy()
train_df['y'] = y_train_log.values

test_df = X_test.copy()
test_df['y'] = y_test_log.values

In [ ]:
def create_sequences(data, target_column='y', window=12):
    X_seq = []
    y_seq = []

    for i in range(window, len(data)):
        X_seq.append(data.iloc[i-window:i].values)
        y_seq.append(data.iloc[i][target_column])

    return np.array(X_seq), np.array(y_seq)

In [ ]:
window_size = 12

X_train_seq, y_train_seq = create_sequences(train_df, window=window_size)
X_test_seq, y_test_seq = create_sequences(test_df, window=window_size)

print("Shape train:", X_train_seq.shape)
print("Shape test:", X_test_seq.shape)

##### Definición del Modelo

In [ ]:
peso_inicial = RandomNormal(mean=0.0, stddev=0.2)

model_lstm = Sequential([
    LSTM(
        64,
        input_shape=(window_size, X_train_seq.shape[2]),
        kernel_initializer=peso_inicial,
        recurrent_initializer=RandomNormal(mean=0.0, stddev=0.1),
        return_sequences=False
    ),
    Dropout(0.3),
    Dense(
        16,
        activation="relu",
        kernel_initializer=RandomNormal(mean=0.0, stddev=0.1)
    ),
    Dense(1)
])

optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

model_lstm.compile(
    optimizer=optimizer,
    loss="mse"
)

model_lstm.summary()

##### Entrenamiento

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

tiempo_inicial = time.time()
history = model_lstm.fit(
    X_train_seq,
    y_train_seq,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)
tiempo_entrenamiento = time.time() - tiempo_inicial

print(f"Tiempo de entrenamiento LSTM: {tiempo_entrenamiento:.2f} segundos")

##### Predicción

In [ ]:
# Predicción
z_pred = model_lstm.predict(X_test_seq).flatten()
y_pred_lstm = np.expm1(z_pred)

# Los primeros 12 puntos de test no se predicen porque no hay secuencia
# completa para esos puntos
y_test_eval = np.expm1(y_test_seq)

##### Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))

ax.plot(y_test.index[window_size:], y_test_eval,
        label='Real', marker='o')

ax.plot(y_test.index[window_size:], y_pred_lstm,
        label='Predicción LSTM', linestyle='--')

ax.set_title("LSTM: Real vs Predicción")
ax.legend()
ax.grid(alpha=0.3)

plt.show()

In [ ]:
# Diagnóstico de residuos para LSTM
residuos_lstm = residuals_diagnostic(y_test_eval, y_pred_lstm, lags=24, titulo="LSTM")

##### Métricas

In [ ]:
metrics_lstm = calculate_metrics(
    y_true=y_test_eval,
    y_pred=y_pred_lstm,
    mase_denom=mase_denom,
    model_name="LSTM"
)
metrics_lstm["Tiempo_Entrenamiento"] = tiempo_entrenamiento

resultados_modelos.append(metrics_lstm)

modelos_entrenados['LSTM'] = {
    'modelo': model_lstm,
    'y_pred': y_pred_lstm
}

print(metrics_lstm)

#### <a class="anchor" id="5_8">5.8 Prophet</a>

**Descripción:** **Prophet** es un modelo de pronóstico de series temporales basado en una **descomposición aditiva** que representa la serie como la suma de tendencia, estacionalidad y ruido. Desarrollado por **Meta**, utiliza funciones flexibles para modelar cambios en la tendencia y componentes estacionales mediante series de Fourier.

In [ ]:
# Unificar en un mismo dataframe el que contiene la variable predictora y la que contiene los regresores (variables independientes)
df_train_prophet = X_train.merge(y_train, how = 'left', left_index = True, right_index = True)
df_test_prophet = X_test.merge(y_test, how = 'left', left_index = True, right_index = True)

# Convertir el índice de la Fecha a una columna
df_train_prophet = df_train_prophet.reset_index()
df_test_prophet = df_test_prophet.reset_index()

# Renombrar columnas por el nombre que el algortimo necesita para trabajar
df_train_prophet = df_train_prophet.rename(columns = {"Fecha": "ds", "Volumenproduccion": "y"})
df_test_prophet = df_test_prophet.rename(columns = {"Fecha": "ds", "Volumenproduccion": "y"})

df_train_prophet.head()

In [ ]:
# Crear el modelo
model_prophet = Prophet(yearly_seasonality = True, weekly_seasonality = False, daily_seasonality = False, changepoint_prior_scale = 0.05, seasonality_prior_scale = 0.01)

# Incluir regresores (variables independientes) en el modelo
model_prophet.add_regressor('Anio')
model_prophet.add_regressor('Sembrada')
model_prophet.add_regressor('Cosechada')
model_prophet.add_regressor('Rendimiento')

# Entrenar el modelo
tiempo_inicial = time.time()
model_prophet.fit(df_train_prophet)
tiempo_entrenamiento = time.time() - tiempo_inicial
print(f"Tiempo de entrenamiento Prophet: {tiempo_entrenamiento:.2f} segundos")

In [ ]:
# Para realizar predicciones sobre el dataframe de prueba, primero hay que crear el dataframe futuro
future = df_test_prophet[['ds', 'Anio', 'Sembrada', 'Cosechada', 'Rendimiento']]
future.head()

In [ ]:
# Realizar predicciones
forecast_prophet = model_prophet.predict(future)

In [ ]:
# Graficar resultados
plt.figure(figsize = (14,6))
plt.plot(df_test_prophet['ds'], df_test_prophet['y'], label = 'Real', marker = 'o', markersize = 3)
plt.plot(forecast_prophet['ds'], forecast_prophet['yhat'], label = 'Predicción', linestyle='--')

plt.fill_between(
    forecast_prophet['ds'],
    forecast_prophet['yhat_lower'],
    forecast_prophet['yhat_upper'],
    alpha = 0.2,
    label = 'Intervalo de confianza'
)

plt.title('Prophet: Real vs Predicción')
plt.legend()
plt.show()

In [ ]:
# Calcular métricas de desempeño para el modelo Prophet
prophet_model_metrics = calculate_metrics(y_true = df_test_prophet['y'], y_pred = forecast_prophet['yhat'], mase_denom = mase_denom, model_name = "Prophet")
prophet_model_metrics["Tiempo_Entrenamiento"] = tiempo_entrenamiento

# Almacenar para comparación
resultados_modelos.append(prophet_model_metrics)

modelos_entrenados['Prophet'] = {
    'modelo': model_prophet,
    'y_pred': forecast_prophet['yhat']
}

prophet_model_metrics

In [ ]:
# Predicciones sobre el conjunto de entrenamiento
prophet_model = modelos_entrenados["Prophet"]["modelo"]
df_train_prophet = pd.DataFrame({
    "ds": y_train.index,
    "Anio": X_train["Anio"].values,
    "Sembrada": X_train["Sembrada"].values,
    "Cosechada": X_train["Cosechada"].values,
    "Rendimiento": X_train["Rendimiento"].values
})
forecast_train = prophet_model.predict(df_train_prophet)
modelos_entrenados["Prophet"]["y_pred_train"] = forecast_train["yhat"].values

### <a class="anchor" id="ensamble">6. Entrenamiento y evaluación de **modelos de ensamble**</a>

A continuación, se construyen diversos modelos de ensamble con el objetivo de integrar en una sola arquitectura las principales fortalezas de los modelos más destacados desarrollados en la fase anterior del proyecto. Para la construcción de dichos ensambles se tomó en consideración el uso de algoritmos que apliquen estrategias de ensamble heterogéneas, ya que con ello se logra construir modelos de ensamble por medio de modelos base de diferente tipo, mejorando así la generalización y obteniendo una mayor robustez ante ruido y cambios estructurales. Para este caso es especialmente conveniente combinar modelos estadísticos y de aprendizaje automático ya que permite capturar distintos patrones al mismo tiempo.

De la misma forma, también se construirán modelos que apliquen estrategias de ensamble homogéneas con el propósito de reducir la varianza de los modelos. Como este tipo de ensamble utiliza el mismo tipo de modelo base pero repetido varias veces, se obtiene una construcción más robusta en donde pequeñas variaciones en los datos no cambian radicalmente el resultado final, logrando así una mayor estabilidad.

#### <a class="anchor" id="6_1">6.1 Modelo de Ensamble: **Promedio simple (Simple Averaging Ensemble)**</a>

**Ensable de agregación simple (bagging-style aggregation)** basado en **promedio aritmético de predicciones**.

##### Descripción

Este método combina las predicciones de varios modelos base calculando el **promedio simple** de sus resultados. Cada modelo contribuye con el mismo peso a la predicción final, por lo que el ensamble actúa como un mecanismo de **suavizado del error individual de los modelos**.

##### Justificación

El promedio simple es uno de los enfoques más utilizados en ensembles debido a su **simplicidad y robustez**. Incluso cuando algunos modelos presentan errores individuales, el promedio puede reducir la varianza global de las predicciones. Además, sirve como **baseline para evaluar si estrategias de combinación más complejas realmente aportan mejoras**.

##### Modelos base utilizados

* GPR (Gaussian Process Regression)
* SARIMAX
* Prophet
* MLP optimizado


##### Entrenamiento y Predicción

In [ ]:
# Se calcula el promedio de las predicciones realizadas por los mejores 4 modelos
modelos_entrenados['Prophet']['y_pred'].index = y_test.index
start = time.time()
simple_averaging_ensemble = (pd.Series(modelos_entrenados['GPR']['y_pred'], index = y_test.index) + modelos_entrenados['SARIMAX']['y_pred'] + modelos_entrenados['Prophet']['y_pred'] + pd.Series(modelos_entrenados['MLP_optimizado']['y_pred'], index = y_test.index)) / 4
training_time = time.time() - start

##### Visualizaciones

In [ ]:
# Visualizar resultados
plt.figure(figsize = (14,6))
plt.plot(y_test, label = 'Real', marker = 'o', markersize = 3, color = 'blue')
plt.plot(y_test.index, simple_averaging_ensemble, label = 'Modelo Ensamble', linestyle='--', color = 'green')
plt.title('Modelo Ensamble: Promedio Simple')
plt.legend()
plt.show()

##### Métricas

In [ ]:
# Obtener métricas
sae_model_metrics = calculate_metrics(y_true = y_test, y_pred = simple_averaging_ensemble, mase_denom = mase_denom, model_name = "Simple Averaging Ensemble")
sae_model_metrics["Tiempo_Entrenamiento"] = training_time
resultados_modelos.append(sae_model_metrics)

sae_model_metrics

#### <a class="anchor" id="6_2">6.2 Modelo de Ensamble: **Promedio ponderado (Weighted Average Ensemble)**</a>

**Ensable de agregación ponderada (weighted aggregation ensemble)**.

##### Descripción

Este método extiende el promedio simple asignando **pesos distintos a cada modelo base** según su desempeño individual. Los modelos que presentan menor error reciben mayor peso en la predicción final.

##### Justificación

No todos los modelos tienen el mismo nivel de precisión. El promedio ponderado permite **priorizar modelos más confiables**, lo cual puede mejorar la precisión del ensamble en comparación con el promedio simple. Esta técnica es especialmente útil cuando existe **heterogeneidad en el desempeño de los modelos base**.

##### Modelos base utilizados

* GPR
* SARIMAX
* Prophet
* MLP


##### Entrenamiento y Predicción

In [ ]:
# Se determinan los pesos de cada modelo dependiendo de su error
best_predictions = pd.concat([pd.DataFrame(modelos_entrenados['GPR']['y_pred'], index = y_test.index), pd.DataFrame(modelos_entrenados['SARIMAX']['y_pred']), pd.DataFrame(modelos_entrenados['Prophet']['y_pred']), pd.DataFrame(modelos_entrenados['MLP_optimizado']['y_pred'], index = y_test.index)], axis = 1)
pesos = np.array([0.70, 0.15, 0.125, 0.025])

# Se realiza el promedio ponderado
start = time.time()
weighted_average_ensemble = best_predictions.dot(pesos)
training_time = time.time() - start

##### Visualizaciones

In [ ]:
# Visualizar resultados
plt.figure(figsize = (14,6))
plt.plot(y_test, label = 'Real', marker = 'o', markersize = 3, color = 'blue')
plt.plot(y_test.index, weighted_average_ensemble, label = 'Modelo Ensamble', linestyle='--', color = 'green')
plt.title('Modelo Ensamble: Promedio Ponderado')
plt.legend()
plt.show()

##### Métricas

In [ ]:
# Obtener métricas
wae_model_metrics = calculate_metrics(y_true = y_test, y_pred = weighted_average_ensemble, mase_denom = mase_denom, model_name = "Weighted Average Ensemble")
wae_model_metrics["Tiempo_Entrenamiento"] = training_time
resultados_modelos.append(wae_model_metrics)

wae_model_metrics

#### <a class="anchor" id="6_3">6.3 Modelo de Ensamble: **Apilamiento con meta-modelo lineal (Stacking Ensemble con Ridge Regression)**</a>

**Ensable de apilamiento (stacking ensemble) con meta-modelo lineal**.

##### Descripción

El stacking consiste en entrenar un **meta-modelo** que aprende cómo combinar las predicciones generadas por los modelos base. En este caso, las predicciones de los modelos base se utilizan como variables de entrada para una **regresión Ridge**, la cual produce la predicción final.

La estructura del modelo es:

Modelos base → Predicciones → Meta-modelo (Ridge) → Predicción final

##### Justificación

A diferencia de los métodos de promedio, el stacking permite que el modelo aprenda **cómo combinar las predicciones de manera óptima**. El uso de Ridge Regression como meta-modelo ayuda a controlar la multicolinealidad entre las predicciones de los modelos base y proporciona una combinación estable y regularizada.

##### Modelos base utilizados

* GPR
* SARIMAX
* Prophet
* MLP

Meta-modelo:

* Ridge Regression

##### Entrenamiento y Predicción

In [ ]:
# Se elimina la primera observación para alinear predicciones
meta_index = y_train.index[1:]

# Construir dataset meta-modelo
X_meta_train = pd.DataFrame({
                              "GPR": modelos_entrenados["GPR"]["y_pred_train"],
                              "SARIMAX": modelos_entrenados["SARIMAX"]["y_pred_train"][1:],
                              "Prophet": modelos_entrenados["Prophet"]["y_pred_train"][1:],
                              "MLP": modelos_entrenados["MLP_optimizado"]["y_pred_train"][1:]
                          }, index=meta_index)

y_meta_train = y_train[1:]

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "alpha": np.logspace(-3, 3, 50)
}

start = time.time()

grid = GridSearchCV(
                    Ridge(),
                    param_grid,
                    cv=tscv,
                    scoring="neg_mean_absolute_error"
                )

grid.fit(X_meta_train, y_meta_train)

training_time = time.time() - start

ridge_meta = grid.best_estimator_

print("Mejor alpha encontrado:", grid.best_params_["alpha"])

In [ ]:
# Pesos aprendidos
print("Intercepto:", ridge_meta.intercept_)
print("\nPesos aprendidos:")

for nombre, peso in zip(X_meta_train.columns, ridge_meta.coef_):
    print(nombre, ":", peso)

In [ ]:
# Dataset test
X_meta_test = pd.DataFrame({
                            "GPR": modelos_entrenados["GPR"]["y_pred"],
                            "SARIMAX": modelos_entrenados["SARIMAX"]["y_pred"],
                            "Prophet": modelos_entrenados["Prophet"]["y_pred"],
                            "MLP": modelos_entrenados["MLP_optimizado"]["y_pred"]
                        }, index=y_test.index)

In [ ]:
# Predicción final
y_pred_stack_ridge = ridge_meta.predict(X_meta_test)

modelos_entrenados["Stacking_Ridge"] = {
                                        "modelo": ridge_meta,
                                        "y_pred": y_pred_stack_ridge
                                    }

##### Visualizaciones

In [ ]:
# Visualización
fig, ax = plt.subplots(figsize=(14,6))
ax.plot(y_test.index, y_test, label="Real", linewidth=2, marker="o", markersize=3)
ax.plot(y_test.index, y_pred_stack_ridge, label="Stacking Ridge", linestyle="--", linewidth=2)
ax.set_title("Stacking Ensemble (Ridge): Real vs Predicción")
ax.set_xlabel("Fecha")
ax.set_ylabel("Producción")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Diagnóstico residuos
residuos_stack_ridge = residuals_diagnostic(
                                            y_test,
                                            y_pred_stack_ridge,
                                            lags=24,
                                            titulo="Stacking Ridge"
                                        )

##### Métricas

In [ ]:
# Métricas
metrics_stack_ridge = calculate_metrics(
                                        y_true=y_test,
                                        y_pred=y_pred_stack_ridge,
                                        mase_denom=mase_denom,
                                        model_name="Stacking_Ridge"
                                    )

metrics_stack_ridge["Tiempo_Entrenamiento"] = training_time

resultados_modelos.append(metrics_stack_ridge)

metrics_stack_ridge

#### <a class="anchor" id="6_4">6.4 Modelo de Ensamble: **Apilamiento con meta-modelo no lineal (Stacking Ensemble con Random Forest)**</a>

**Ensable de apilamiento (stacking ensemble) con meta-modelo basado en árboles**.

##### Descripción

En este enfoque se utiliza un **Random Forest** como meta-modelo para combinar las predicciones generadas por los modelos base. A diferencia del stacking lineal, el Random Forest puede capturar **relaciones no lineales entre las predicciones de los modelos base**.

La estructura es:

Modelos base → Predicciones → Meta-modelo (Random Forest) → Predicción final

##### Justificación

Las relaciones entre las predicciones de diferentes modelos pueden ser complejas y no necesariamente lineales. El uso de Random Forest como meta-modelo permite capturar **interacciones y patrones no lineales** entre los modelos base, lo que puede mejorar el desempeño del ensamble en escenarios donde los errores de los modelos no siguen una estructura lineal simple.

##### Modelos base utilizados

* GPR
* SARIMAX
* Prophet
* MLP

Meta-modelo:

* RandomForestRegressor

##### Entrenamiento y Predicción

In [ ]:
# Tiempo al iniciar entrenamiento
inicio = time.time()

# índice correcto (GPR pierde el primer punto)
meta_index = y_train.index[1:]

# Construir dataset meta-modelo
X_meta_train = pd.concat([
    pd.Series(modelos_entrenados["GPR"]["y_pred_train"], index=meta_index, name="GPR"),
    pd.Series(modelos_entrenados["GPR"]["y_std_train"], index=meta_index, name="GPR_std"),
    pd.Series(modelos_entrenados["SARIMAX"]["y_pred_train"], index=y_train.index, name="SARIMAX"),
    pd.Series(modelos_entrenados["Prophet"]["y_pred_train"], index=y_train.index, name="Prophet"),
    pd.Series(modelos_entrenados["MLP_optimizado"]["y_pred_train"], index=y_train.index, name="MLP_optimizado"),
    pd.Series(modelos_entrenados["SVR"]["y_pred_train"], index=y_train.index, name="SVR")
], axis=1).dropna()

# Target alineado
y_meta_train = y_train.loc[X_meta_train.index]

# Meta features adicionales: estacionalidad
X_meta_train["mes"] = X_meta_train.index.month
X_meta_train["sin_mes"] = np.sin(2*np.pi*X_meta_train["mes"]/12)
X_meta_train["cos_mes"] = np.cos(2*np.pi*X_meta_train["mes"]/12)
X_meta_train["lag1"] = y_meta_train.shift(1)

# dispersión entre modelos
X_meta_train["std_modelos"] = X_meta_train[
["GPR","SARIMAX","Prophet","MLP_optimizado","SVR"]
].std(axis=1)

# ---------------------

# Test
X_meta_test = pd.DataFrame({
    "GPR": modelos_entrenados["GPR"]["y_pred"],
    "GPR_std": modelos_entrenados["GPR"]["y_std"],
    "SARIMAX": modelos_entrenados["SARIMAX"]["y_pred"],
    "Prophet": modelos_entrenados["Prophet"]["y_pred"],
    "MLP_optimizado": modelos_entrenados["MLP_optimizado"]["y_pred"],
    "SVR": modelos_entrenados["SVR"]["y_pred"]
}, index=y_test.index)

X_meta_test["mes"] = X_meta_test.index.month
X_meta_test["sin_mes"] = np.sin(2*np.pi*X_meta_test["mes"]/12)
X_meta_test["cos_mes"] = np.cos(2*np.pi*X_meta_test["mes"]/12)
X_meta_test["lag1"] = y_test.shift(1)

X_meta_test["std_modelos"] = X_meta_test[
["GPR","SARIMAX","Prophet","MLP_optimizado","SVR"]
].std(axis=1)

# Pesos para el stacking, inversamente proporcionales al MASE
modelos_objetivo = ['GPR', 'SARIMAX', 'Prophet', 'MLP_optimizado', 'SVR']
pesos = {}

for res in resultados_modelos:
    nombre = res['Modelo']
    if nombre in modelos_objetivo:
        valor_mase = res['MASE']
        # Evitamos error si MASE es 0
        if valor_mase != 0:
            pesos[nombre] = 1 / valor_mase

suma = sum(pesos.values())
pesos = {k:v/suma for k,v in pesos.items()}

for m in ["GPR","SARIMAX","Prophet","MLP_optimizado","SVR"]:
    X_meta_train[m] *= pesos[m]
    X_meta_test[m] *= pesos[m]

features_meta = [
"GPR",
"GPR_std",
"SARIMAX",
"Prophet",
"MLP_optimizado",
"SVR",
"sin_mes",
"cos_mes",
"std_modelos",
"lag1"
]

In [ ]:
# Validación cruzada para stacking
tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
# Ajuste de hiperparámetros para Random Forest
param_dist = {
    "bootstrap": [True, False],
    "max_depth": [4,6,8,10,12,None],
    "max_features": ["sqrt","log2",0.5,0.7],
    "max_samples": [None,0.6,0.7,0.8,0.9],
    "min_samples_leaf": [1,2,3,4,6],
    "min_samples_split": [2,4,6,8,10],
    "n_estimators": [300,500,800,1200]
}

# Definir Random Forest
rf = RandomForestRegressor(random_state=42)

# Búsqueda aleatoria con validación cruzada
search = RandomizedSearchCV(
    rf,
    param_dist,
    n_iter=80,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

# Búsqueda de los mejores hiperparámetros
search.fit(X_meta_train[features_meta], y_meta_train)
rf_meta = search.best_estimator_

print("Mejores hiperparámetros:", search.best_params_)

In [ ]:
# Predicción final
rf_meta.fit(X_meta_train[features_meta], y_meta_train)
y_pred_stack_rf = rf_meta.predict(X_meta_test[features_meta])

tiempo_entrenamiento = time.time() - inicio
print("Tiempo entrenamiento:", tiempo_entrenamiento)

##### Visualizaciones

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))

ax.plot(
    y_test.index,
    y_test,
    label="Real",
    linewidth=2,
    marker="o",
    markersize=3
)

ax.plot(
    y_test.index,
    y_pred_stack_rf,
    label="Stacking RF",
    linestyle="--",
    linewidth=2
)

ax.set_title("Stacking Ensemble (Random Forest): Real vs Predicción")
ax.set_xlabel("Fecha")
ax.set_ylabel("Producción (normalizada)")

ax.legend()
ax.grid(alpha=0.3)

plt.show()

In [ ]:
residuos_stack_rf = residuals_diagnostic(
    y_test,
    y_pred_stack_rf,
    lags=24,
    titulo="Stacking RF"
)

##### Métricas

In [ ]:
metrics_stack_rf = calculate_metrics(
    y_true=y_test,
    y_pred=y_pred_stack_rf,
    mase_denom=mase_denom,
    model_name="Stacking_RF"
)
metrics_stack_rf["Tiempo_Entrenamiento"] = tiempo_entrenamiento

resultados_modelos.append(metrics_stack_rf)
print(metrics_stack_rf)

modelos_entrenados["Stacking_RF"] = {
    "modelo": rf_meta,
    "y_pred": y_pred_stack_rf
}

#### <a class="anchor" id="6_5">6.5 Modelo de Ensamble: **Bagging con GPR (Ensamble Homogéneo)**</a>

##### Descripción

Este modelo implementa un ensamble homogéneo tipo Bagging (Bootstrap Aggregating) utilizando múltiples instancias de GPR, disponibles en la biblioteca Scikit-learn.

El método consiste en entrenar varios modelos del mismo tipo sobre muestras bootstrap del conjunto de entrenamiento. Cada modelo produce una predicción independiente y el resultado final del ensamble se obtiene promediando las predicciones individuales.

##### Justificación

El Bagging es particularmente útil cuando el modelo base presenta alta varianza, como puede ocurrir con los GPR al ajustarse a datos complejos o ruidosos. Sus principales ventajas son:

- Reduce la varianza del modelo sin incrementar significativamente el sesgo.
- Mejora la robustez frente a ruido en los datos de entrenamiento.
- Permite aprovechar la capacidad probabilística del GPR manteniendo interpretabilidad.

En problemas de series de tiempo, el Bagging puede ayudar a estabilizar predicciones cuando los patrones temporales presentan variabilidad o cambios estructurales.

##### Modelos base utilizados

El ensamble utiliza como modelo base GPR, seleccionado por haber mostrado el mejor desempeño entre los modelos evaluados previamente. Cada miembro del ensamble mantiene:

- el mismo kernel
- los mismos hiperparámetros
- diferentes submuestras bootstrap del conjunto de entrenamiento

##### Entrenamiento y Predicción

In [ ]:
# Número de modelos del ensamble
n_estimators = 10

gpr_base = modelos_entrenados['GPR']['modelo']
kernel = gpr_base.kernel.clone_with_theta(
    gpr_base.kernel.theta + np.random.normal(0,0.05,len(gpr_base.kernel.theta))
)

bagging_models = []
preds = []
stds = []

start_time = time.time()

for i in range(n_estimators):

    # Bootstrap sample
    X_boot, y_boot = resample(
        X_train_gpr,
        y_train_gpr,
        replace=True,
        random_state=42 + i
    )

    # Crear modelo con mismo kernel
    gpr = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=10,
        random_state=42 + i
    )

    # Entrenar
    gpr.fit(X_boot, y_boot)

    # Predecir
    y_pred, y_std = gpr.predict(X_test_gpr, return_std=True)

    bagging_models.append(gpr)
    preds.append(y_pred)
    stds.append(y_std)

tiempo_entrenamiento = time.time() - start_time

In [ ]:
# Predicciones
preds = np.array(preds)
stds = np.array(stds)

# Promedio de predicciones
y_pred_bagging = preds.mean(axis=0)

# Promedio de incertidumbre
y_std_bagging = stds.mean(axis=0)

##### Visualizaciones

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))

ax.plot(
    y_test.index,
    y_test,
    label="Real",
    linewidth=2,
    marker="o",
    markersize=3
)

ax.plot(
    y_test.index,
    y_pred_stack_rf,
    label="Stacking RF",
    linestyle="--",
    linewidth=2
)

ax.set_title("Stacking Ensemble (Random Forest): Real vs Predicción")
ax.set_xlabel("Fecha")
ax.set_ylabel("Producción (normalizada)")

ax.legend()
ax.grid(alpha=0.3)

plt.show()

In [ ]:
residuos_stack_rf = residuals_diagnostic(
    y_test,
    y_pred_stack_rf,
    lags=24,
    titulo="Stacking RF"
)

##### Métricas

In [ ]:
metrics_gpr_bagging = calculate_metrics(
    y_true=y_test_gpr,
    y_pred=y_pred_bagging,
    mase_denom=mase_denom,
    model_name="Bagging_GPR"
)
metrics_gpr_bagging["Tiempo_Entrenamiento"] = tiempo_entrenamiento

resultados_modelos.append(metrics_gpr_bagging)

modelos_entrenados['Bagging_GPR'] = {
    'modelo': bagging_models,
    'y_pred': y_pred_bagging,
    'y_std': y_std_bagging
}

print(metrics_gpr_bagging)

### <a class="anchor" id="comparacion">7. Comparación global de modelos</a>

Se seleccionó **MASE como métrica principal** debido a que es una medida escalable y robusta para problemas de series temporales. MASE compara el error del modelo contra un pronóstico ingenuo (Exponential Smoothing), permitiendo interpretar directamente si el modelo mejora el desempeño de una predicción simple. Valores menores a 1 indican que el modelo supera al baseline.

Para comparar los modelos se construyó un ranking basado en un **score ponderado de métricas normalizadas**.

Se asignó mayor peso a **MASE (50%)**, por ser la métrica principal. Métricas adicionales como **RMSE (15%)** y **MAE (15%)** capturan la magnitud del error absoluto y cuadrático, mientras que **MAPE (10%)** y **R² (10%)** aportan información complementaria sobre error relativo y capacidad explicativa del modelo.

Las métricas fueron normalizadas mediante **min-max scaling** para hacerlas comparables antes de calcular el score final.

#### 7.1 Ranking de Modelos

In [ ]:
# Convertir a DataFrame
df_metricas = pd.DataFrame(resultados_modelos)

# Copia para trabajar
df_rank = df_metricas.copy()

# Convertir R² a error
df_rank['R2_error'] = 1 - df_rank['R²']

# Métricas a usar
metricas = ['MAE', 'RMSE', 'MAPE', 'MASE', 'R2_error']

# Pesos (MASE dominante)
pesos = {
    'MAE': 0.15,
    'RMSE': 0.15,
    'MAPE': 0.10,
    'MASE': 0.50,
    'R2_error': 0.10
}

# Normalización min-max
for m in metricas:

    min_m = df_rank[m].min()
    max_m = df_rank[m].max()

    if max_m - min_m == 0:
        df_rank[m + '_norm'] = 0
    else:
        df_rank[m + '_norm'] = (df_rank[m] - min_m) / (max_m - min_m)

# Score ponderado
df_rank['Score_final'] = 0

for m in metricas:
    df_rank['Score_final'] += pesos[m] * df_rank[m + '_norm']

# Ordenar ranking
df_rank = df_rank.sort_values('Score_final')

ranking_final = df_rank[['Modelo','Score_final','MASE','RMSE','MAE','MAPE','R²','Tiempo_Entrenamiento']].reset_index(drop=True)

ranking_final

#### 7.2 Visualización comparativa

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=df_rank,
    x='Score_final',
    y='Modelo',
    palette='viridis'
)

plt.title("Ranking global de modelos (Score ponderado)")
plt.xlabel("Score final (menor es mejor)")
plt.ylabel("Modelo")

plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.show()

In [ ]:
# Convertir a DataFrame
df_radar = pd.DataFrame(resultados_modelos)

# Top-5 y baseline
top5 = ranking_final.head(5)
baseline = "Exponential Smoothing"

# Unir
if baseline not in top5["Modelo"].values:
    baseline_row = df_radar[df_radar["Modelo"] == baseline]
    df_plot = pd.concat([top5, baseline_row])
else:
    df_plot = top5.copy()

# Métricas a usar
metrics = ['MAE', 'RMSE', 'MAPE', 'MASE', 'R²']
df_plot = df_plot[["Modelo"] + metrics].reset_index(drop=True)

# Invertir métricas de error (menor = mejor)
for m in ['MAE','RMSE','MAPE','MASE']:
    df_plot[m] = 1 / df_plot[m]

# Normalizar métricas entre 0 y 1
df_norm = df_plot.copy()
for m in metrics:
    min_val = df_plot[m].min()
    max_val = df_plot[m].max()
    df_norm[m] = (df_plot[m] - min_val) / (max_val - min_val)

# Preparar radar
labels = metrics
num_vars = len(labels)
linestyles = ['-', '--', ':', '-.', '-', '--']
markers = ['o', 's', 'D', '^', 'v', 'P']
colors = {
    "Stacking_RF": "#1f77b4",
    "Stacking_Ridge": "#d62728",
    "GPR": "#2ca02c",
    "Weighted Average Ensemble": "#ff7f0e",
    "Bagging_GPR": "#9467bd",
    "Exponential Smoothing": "#000000"
}

angles = np.linspace(0, 2*np.pi, num_vars, endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

fig, ax = plt.subplots(figsize=(12,12), subplot_kw=dict(polar=True))

for i, row in df_norm.iterrows():

    values = row[metrics].values
    values = np.concatenate([values, [values[0]]])
    
    if row['Modelo'] == 'Exponential Smoothing':
        ax.plot(
            angles,
            values,
            linewidth=3,
            linestyle=':',
            color='black',
            marker='X',
            label=row['Modelo']
        )
    else:
        ax.plot(
            angles,
            values,
            linewidth=2,
            linestyle=linestyles[i],
            marker=markers[i],
            color=colors[row["Modelo"]],
            label=row['Modelo']
        )

ax.set_thetagrids(angles[:-1] * 180/np.pi, labels)

plt.title("Comparación de Modelos (mayor es mejor)", size=14)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.show()

In [ ]:
top3 = ranking_final['Modelo'].head(3).tolist()
top3

In [ ]:
final_colors = [
    "#1f77b4",
    "#d62728",
    "#2ca02c"
]

plt.figure(figsize=(14,6))

plt.plot(y_test.index, y_test, label='Real', linewidth=3)

for i, modelo in enumerate(top3):

    y_pred = modelos_entrenados[modelo]['y_pred']

    plt.plot(
        y_test.index,
        y_pred,
        label=modelo,
        linestyle=linestyles[i],
        color=final_colors[i]
    )

plt.title("Comparación de predicciones (Top 3 modelos)")
plt.xlabel("Fecha")
plt.ylabel("Producción")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

#### 7.3 Selección del modelo final

Como se puede observar en la tabla comparativa, el mejor modelo construido es el modelo de ensamble Stacking_RF. Primeramente destaca por la métrica del `Score_final` el cual ponderaba todas las métricas empleadas y las normalizaba, y aquí se puede observar que es el modelo que obtuvo el menor error posible, volviéndolo el más confiable que se haya construido en este proyecto. Dentro de esta evaluación, la métrica `MASE` es nuestra medida principal, y en dicha tabla se puede observar que el modelo Stacking_RF es el que tiene la medida más baja de todos los modelos, teniendo 0.48, interpretándose así como el modelo con mejor desempeño en comparación con el modelo de referencia Exponential Smoothing.

Tomando en cuenta de que se está trabajando con un problema de series de tiempo y que los objetivos del negocio se encaminan a estimar el volumen de producción, entonces se puede complementar la elección del modelo final con otras 2 métricas particulares. La primera métrica, `RMSE`, es muy importante para estas necesidades del negocio, porque como penaliza significativamente los errores grandes cometidos por el modelo, entonces permite evaluar la robustez para evitar que se cometan errores costosos para el negocio. En la tabla comparativa, se puede ver que el modelo Stacking_RF es el que tiene el menor RMSE, teniendo un error típico de aproximadamente 0.05 en la escala normalizada de volumen de producción de aguacate.

La segunda métrica, `MAE`, es importante debido a que facilita la interpretación de los resultados del modelo y evalúa su desempeño de manera intuitiva empleando las unidades de la variable objetivo. En la tabla comparativa, nuevamente se puede observar que el modelo Stacking_RF es el que tiene el menor MAE, donde en promedio el modelo se equivoca 0.04 unidades en la escala normalizada del volumen de producción de aguacate, es decir un 4% del rango total. Como la métrica obtenida de RMSE es muy similar a la de MAE, entonces se puede saber que los errores del modelo son bastantes consistentes y no hay errores grandes ocasionales.

Adicionalmente, también se empleó la métrica `MAPE`, pero como se está trabajando con datos normalizados, para que no se rompiera su cálculo se tuvieron que omitir registros cuyo valor fuera igual a cero, por lo que el resultado de esta métrica distorsiona la interpretación real del modelo en este caso, pero de igual forma es útil como una referencia adicional. En la tabla comparativa se puede observar que el modelo Stacking_RF nuevamente obtuvo la medida más baja para la métrica MAPE, indicando de forma sesgada que en promedio el modelo se equivoca un 6.29% respecto al valor real de producción.

Por último, también se empleó la métrica `R2` que si bien, para modelos que trabajan con series de tiempo no es apropiado porque pierde significado al comparar el modelo contra una predicción basada en la medida global, sí puede ser empleado como criterio de descarte. Como se puede observar en la tabla comparativa, esta vez el modelo Stacking_RF no tiene la medida más baja de la métrica R2, pero eso no es de utilidad para este análisis; lo que sí es importante es que su valor de 0.68 es positivo, porque un valor de R2 negativo en el conjunto de prueba indicaría que el modelo estaría subajustado

En cuanto al tiempo de entrenamiento, si bien este modelo es uno de los que más se tarda en entrenar, analizándolo individualmente se puede saber que es un tiempo más que adecuado por los resultados obtenidos, y es que el modelo tarda poco menos de un minuto en ser entrenado, exactamente 58.60 segundos, lo cual lo vuelve completamente viable para su construcción y futuros reajustes. 

Tomando en cuenta todo lo comentado previamente y considerando las necesidades y objetivos del negocio, se puede definir que **el modelo final seleccionado es el ensamble Stacking Random Forest (Stacking_RF)**. Este modelo no solamente superó a todos los modelos entrenados individualmente y al resto de ensambles, sino que también las métricas reflejan su capacidad para poder ser de confianza y utilidad para el negocio, para la predicción mensual del volumen de producción de aguacate en el estado de Jalisco.


### <a class="anchor" id="interpretacion">8. Interpretación del modelo final (Stacking Random Forest)</a>

En esta sección se presentan diversas visualizaciones del modelo Stacking Random Forest con el objetivo de analizar su desempeño desde diferentes perspectivas. A través de gráficos de predicción, errores y análisis de residuos, se busca evaluar la calidad de las predicciones, identificar posibles patrones no capturados por el modelo y comprender qué modelos o variables influyen más en el ensamble.

In [ ]:
# Predicciones del modelo ganador
y_pred = modelos_entrenados["Stacking_RF"]["y_pred"]
modelo_stack = modelos_entrenados["Stacking_RF"]["modelo"]

# Residuos
residuos = y_test - y_pred

##### Serie temporal (Real vs Predicción)



In [ ]:
plt.figure(figsize=(12,6))

plt.plot(y_test.index, y_test, label="Real", linewidth=2)
plt.plot(y_test.index, y_pred, label="Stacking RF", linewidth=2)

plt.title("Serie Temporal: Valores Reales vs Predicción (Stacking RF)")
plt.xlabel("Fecha")
plt.ylabel("Valor")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

**Análsis**

El gráfico muestra la comparación entre los valores reales de la serie y las predicciones generadas por el modelo Stacking Random Forest.

Se observa que el modelo logra capturar de manera adecuada los patrones generales de la serie temporal, incluyendo las fluctuaciones periódicas y la tendencia global del fenómeno. En la mayoría de los periodos, la predicción sigue de cerca el comportamiento de los valores reales.

Sin embargo, también se identifican desviaciones puntuales, especialmente en periodos donde ocurren caídas o incrementos abruptos en la serie real. Estas discrepancias sugieren la presencia de eventos extraordinarios o factores externos que no están representados en las variables explicativas del modelo, tales como condiciones climáticas, factores económicos, decisiones políticas o eventos sociales.

En general, el modelo presenta una capacidad adecuada de aproximación a la dinámica de la serie, reproduciendo la estructura principal del comportamiento observado.

##### Residuos vs Tiempo

In [ ]:
plt.figure(figsize=(12,5))

plt.scatter(y_test.index, residuos)

plt.axhline(0, linestyle="--")

plt.title("Residuos del Modelo (Stacking RF)")
plt.xlabel("Fecha")
plt.ylabel("Residuo")

plt.grid(alpha=0.3)

plt.show()

**Análsis**

El gráfico de residuos muestra la diferencia entre los valores reales y las predicciones del modelo a lo largo del tiempo.

Se observa que los residuos se distribuyen alrededor de la línea cero sin presentar una tendencia clara o patrones sistemáticos en el tiempo. Esto sugiere que el modelo no presenta sesgos evidentes, ya que los errores positivos y negativos aparecen de forma relativamente equilibrada.

La ausencia de una estructura visible en los residuos indica que el modelo ha capturado la mayor parte de la información sistemática presente en la serie. No obstante, algunos errores de mayor magnitud aparecen en ciertos periodos específicos, lo cual puede estar asociado a variaciones abruptas en la serie real que el modelo no logra anticipar completamente.

En términos generales, el patrón observado en los residuos es consistente con un modelo que explica adecuadamente la tendencia y la estacionalidad de la serie, aunque todavía existen fluctuaciones no modeladas.

##### Histograma de Residuos

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(residuos, bins=15)

plt.axvline(0, linestyle="--")

plt.title("Distribución de Residuos")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")

plt.grid(alpha=0.3)

plt.show()

**Análsis**

El histograma de residuos permite analizar la distribución estadística de los errores del modelo.

Se observa que la distribución de los residuos se encuentra centrada alrededor de cero, lo cual es deseable en modelos de predicción, ya que indica que el modelo no presenta un sesgo sistemático hacia la sobreestimación o subestimación.

La forma de la distribución se aproxima a una distribución aproximadamente normal, aunque se aprecia una ligera asimetría y la presencia de algunas observaciones en las colas. Estas colas sugieren la existencia de errores relativamente grandes en ciertos periodos, posiblemente asociados a eventos atípicos o cambios estructurales en la serie.

A pesar de estas desviaciones, la distribución general de los residuos puede considerarse aceptable para un modelo predictivo aplicado a una serie temporal con posibles factores externos no observados.

##### Función de Autocorrelación

In [ ]:
plot_acf(residuos, lags=20)

plt.title("Autocorrelación de Residuos (Stacking RF)")

plt.show()

**Análsis**

El gráfico de autocorrelación muestra que la mayoría de los retardos se encuentran dentro del intervalo de confianza, lo que sugiere que los residuos del modelo son mayoritariamente aleatorios.

Sin embargo, el lag 1 presenta una autocorrelación ligeramente significativa, lo que indica que todavía existe cierta dependencia temporal no capturada completamente por el modelo.

En general, el comportamiento observado es cercano al de ruido blanco, aunque la presencia de autocorrelación en los primeros lags sugiere que aún podrían existir patrones temporales menores sin modelar.

##### Contribución de Modelos Base y Variables al Ensamble

In [ ]:
importancias = pd.Series(
    modelo_stack.feature_importances_,
    index=features_meta
).sort_values(ascending=False)

plt.figure(figsize=(8,5))

importancias.plot(kind="bar")

plt.title("Importancia de Variables en Stacking Random Forest")
plt.ylabel("Importancia")

plt.grid(axis="y", alpha=0.3)

plt.show()

**Análsis**

El gráfico de importancia muestra una clara dominancia del modelo GPR, que concentra la mayor parte de la contribución en el ensamble. Esto indica que el meta-modelo depende principalmente de sus predicciones para generar el resultado final.

Otros modelos, como MLP optimizado, SARIMAX y Prophet, aportan información adicional pero con una influencia considerablemente menor.

Las variables auxiliares (lag1, sin_mes, cos_mes, desviaciones estándar y SVR) presentan importancia prácticamente nula, lo que sugiere que los patrones temporales y estacionales ya están siendo capturados por los modelos base.

##### Dispersión (Real vs Predicción)

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(y_test, y_pred)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("Valor Real")
plt.ylabel("Predicción")

plt.title("Real vs Predicción (Stacking RF)")

plt.grid(alpha=0.3)

plt.show()

**Análsis**

El gráfico muestra que las predicciones siguen razonablemente la línea ideal `y=x`, lo que indica que el modelo captura la tendencia general de los datos.

Sin embargo, se observa un sesgo en los extremos: el modelo tiende a sobreestimar valores bajos y subestimar valores altos, un comportamiento común en modelos basados en Random Forest, que suelen suavizar las predicciones hacia la media.

La dispersión de los puntos en el rango medio sugiere que el modelo no captura completamente toda la variabilidad, lo que coincide con los patrones residuales observados en el análisis de autocorrelación.

##### Gráfico de Error Acumulado

In [ ]:
# Calcular residuos
residuos = y_test - y_pred_stack_rf

# Error acumulado
error_acumulado = residuos.cumsum()

plt.figure(figsize=(10,5))

plt.plot(error_acumulado, linewidth=2)

plt.axhline(0, linestyle="--", linewidth=1)

plt.title("Error Acumulado del Modelo Stacking_RF")
plt.xlabel("Tiempo")
plt.ylabel("Error acumulado")

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Análsis**

El error acumulado muestra diferentes fases en el desempeño del modelo a lo largo del tiempo.

Entre 2019 y 2022 el error se mantiene relativamente bajo y estable, lo que indica un buen ajuste general. Sin embargo, a partir de finales de 2022 aparece una tendencia creciente del error, alcanzando su punto máximo alrededor de 2024, lo que sugiere que el modelo tuvo mayores dificultades para adaptarse a cambios en el comportamiento de la serie.

Este comportamiento podría estar relacionado con la alta dependencia del modelo GPR, que domina el ensamble y puede tener limitaciones para capturar cambios más abruptos en la dinámica de los datos.

##### Error Absoluto en el Tiempo

In [ ]:
error_abs = np.abs(residuos)

plt.figure(figsize=(10,5))

plt.plot(error_abs, linewidth=2)

plt.title("Error Absoluto a lo largo del tiempo - Stacking_RF")
plt.xlabel("Tiempo")
plt.ylabel("|Error|")

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Análsis**

El error absoluto muestra que, aunque el modelo suele tener errores pequeños, existen picos ocasionales de error elevado.

Se observan picos importantes al inicio del periodo analizado y nuevamente entre 2022 y 2024, lo que indica momentos donde el modelo pierde precisión temporalmente. Estos picos coinciden con el aumento del error acumulado, lo que confirma que durante ese periodo los errores fueron más frecuentes y de mayor magnitud.

En conjunto, el gráfico sugiere que el modelo es generalmente preciso, pero presenta episodios de mayor error cuando cambia el comportamiento de la serie.

##### Diagnóstico General del Modelo

El modelo de Stacking Random Forest logra capturar adecuadamente la tendencia general de la serie temporal, como se observa en la relación entre valores reales y predicciones. Sin embargo, presenta cierto sesgo en los extremos, tendiendo a sobreestimar valores bajos y subestimar valores altos, un comportamiento típico de modelos basados en árboles que suavizan las predicciones hacia la media.

El análisis de errores muestra que, aunque el modelo mantiene errores generalmente bajos, existen episodios puntuales de mayor desviación, especialmente entre 2022 y 2024. Esto se refleja tanto en los picos del error absoluto como en el incremento del error acumulado durante ese periodo.

Por otra parte, el análisis de autocorrelación de los residuos indica la presencia de ligera autocorrelación en los primeros retardos, lo que sugiere que aún quedan algunos patrones temporales sin capturar completamente. Finalmente, el análisis de importancia de variables revela que el ensamble depende en gran medida del modelo GPR, mientras que otros modelos y variables aportan contribuciones menores.

En conjunto, estos resultados indican que el ensamble funciona eficazmente como un refinador del modelo dominante, logrando buenas predicciones globales, aunque con margen de mejora para capturar cambios abruptos en la dinámica temporal de la serie.

### <a class="anchor" id="conclusiones">Conclusiones</a>

Partiendo del avance 4, en donde se exploraron varios modelos alternativos y se realizó una comparación entre estos, podemos rescatar que los mejores resultados obtenidos fueron de GPR, SARIMAX, Prophet y MLP optimizado. Específicamente, el modelo GPR destacó claramente sobre los demás, alcanzando un `MAE` de 0.0427, un `RMSE` de 0.0562, un `R2` de 0.6765 y un `MASE` de 0.4939, siendo este último importante ya que compara el desempeño contra un modelo baseline. Estos valores indican que GPR, en comparación con los otros modelos, logró reducir el error absoluto y cuadrático, además de demostrar una alta capacidad para capturar patrones no lineales en la serie temporal, lo que lo posicionó como el modelo individual más fuerte.
En el caso de SARIMAX, los resultados mostraron un desempeño sólido, con un `MASE` de 0.5781. Por su parte, Prophet y MLP optimizado mostraron buenos desempeños, pero ligeramente inferiores. Aunque ambos lograron capturar tendencia y estacionalidad, sus métricas de error fueron mayores en comparación con GPR.


Dado que GPR fue el modelo individual más fuerte, en esta nueva etapa se buscó cumplir con el objetivo de mejorar significativamente el rendimiento, aprovechando las fortalezas de distintos modelos y reduciendo sus debilidades. Para ello, se construyeron cinco estrategias de ensamble, tanto homogéneas como heterogéneas, las cuales fueron: Promedio simple (Simple Averaging Ensemble), que es un ensamble de agregación directa basado en el promedio aritmético de las predicciones, el Promedio ponderado (Weighted Average Ensemble), que es un ensamble de agregación ponderada donde cada modelo contribuye según su desempeño, el Stacking con meta-modelo lineal (Stacking + Ridge), que es un ensamble heterogéneo donde un modelo lineal aprende a combinar las predicciones, el Stacking con meta-modelo no lineal (Stacking + Random Forest), que es un ensamble heterogéneo más complejo, capaz de capturar relaciones no lineales entre modelos base, y el Bagging con GPR, que es un ensamble homogéneo que genera múltiples instancias del mismo modelo mediante bootstrap.


Al analizar el ranking global basado en el score ponderado con `MASE` como métrica principal, se observa que los modelos de stacking (heterogéneos) superaron tanto a los modelos individuales como al ensamble homogéneo. En particular, **el modelo Stacking con Random Forest obtuvo el mejor desempeño global**, con un `MASE` de 0.4851, `RMSE` de 0.0555, `MAE` de 0.0420 y `R2` de 0.6856, posicionándose en el primer lugar del ranking. Específicamente, por medio del gráfico “Comparación de Modelos (mayor es mejor)”, se puede observar de manera visual el desempeño de los principales modelos, los cuales simulan simultáneamente todas las métricas relevantes, previamente normalizadas para que sean comparables bajo el criterio mayor es mejor. Como ya se mencionó, el modelo Stacking_RF cuenta con la mejor evaluación, ligeramente superior al resto, aunque sin una brecha muy amplia. Seguido de este se encuentran Stacking_Ridge y GPR, que presentan resultados prácticamente superpuestos en casi todos los ejes. De igual forma, el resto de los modelos, debido a su ubicación en el gráfico, indica que su rendimiento es muy similar, ya que ninguno presenta una debilidad marcada en alguna métrica específica y las diferencias entre ellos son pequeñas y consistentes.


Cabe destacar que el modelo GPR individual se mantuvo en tercera posición, con métricas prácticamente equivalentes a Stacking_Ridge. Esto significa que GPR ya era un modelo altamente efectivo por sí solo y que, gracias al ensamble, fue posible ajustar ligeramente su capacidad predictiva. Asimismo, el análisis de importancia de variables mostró que el meta-modelo se apoya principalmente en las predicciones de GPR. Esto sugiere que el stacking no reemplaza al modelo base más fuerte, sino que lo utiliza como referencia principal y realiza pequeños ajustes para corregir desviaciones puntuales.
Entre los principales motivos por los cuales el modelo Stacking Random Forest presenta las mejores métricas se encuentra la combinación de modelos con enfoques distintos y el uso de un meta-modelo basado en árboles, el cual puede capturar relaciones no lineales entre predicciones. De esta manera, se reducen debilidades individuales al aprovechar fortalezas complementarias, lo que indica que esta mayor complejidad estructural se traduce en una mejora en la estabilidad ante errores y en la precisión en la generalización de los datos.



De manera adicional, los gráficos generados permiten confirmar que el modelo captura adecuadamente la tendencia y estacionalidad, y que, aunque suaviza extremos, que es comportamiento propio de modelos basados en árboles, mantiene un ajuste consistente. Presenta residuos cercanos a cero, una distribución de errores aproximadamente normal, muestra autocorrelación mínima en los residuos y errores acumulados controlados, excepto en periodos específicos de alta variabilidad. En conjunto, estos resultados indican que el modelo generaliza adecuadamente y no muestra evidencia clara de sobreajuste ni de subajuste.


En conclusión, la implementación de modelos de ensamble permitió cumplir con el objetivo de mejorar el desempeño respecto a los modelos individuales. Los enfoques de stacking heterogéneo demostraron ser superiores a los ensambles homogéneos y a los modelos individuales. El modelo Stacking Random Forest fue seleccionado como modelo final debido a su menor `MASE`, mejora consistente en las métricas principales y adecuada generalización en datos no vistos, además de contar con un tiempo de entrenamiento moderado. Si bien el modelo cumple con los criterios de éxito establecidos, siempre existe margen de mejora mediante validaciones temporales más extensas y monitoreo continuo, con el objetivo de preservar su estabilidad frente a posibles cambios estructurales futuros.


### <a class="anchor" id="ref">Referencias</a>

Arize. (2023). R-squared — Understanding the coefficient of determination. Arize AI Blog. https://arize.com/blog-course/r-squared-understanding-the-coefficient-of-determination/

Smith, A. (2023). The foundation of model evaluation. Medium. https://medium.com/@audleysmith876/1-0-the-foundation-of-model-evaluation-81a3946ed228